# 📓 Notebook 17 — AI Document Processing

> **Module:** AI Engineering · **Estimated time:** 60–75 min · **Difficulty:** Intermediate

A *huge* fraction of business work is extracting structured information from unstructured documents — invoices, receipts, contracts, lease agreements, KYC forms. Before LLMs, this was either *manual data entry* or *brittle regex*. With LLMs, it's a few dozen lines of Python.

This notebook walks through the full pipeline:

1. **Extract** text from PDFs (or text files).
2. **Chunk** long documents into model-sized pieces.
3. **Use an LLM** to pull structured fields out of each chunk.
4. **Validate** the output with JSON Schema and per-field type checks.
5. **Aggregate** the results into a pandas DataFrame ready for analysis.

By the end you will have built a tiny "invoice parser" that turns a folder of mock invoices into a tidy table of (vendor, total, due_date, line_items) records.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Generate text from a PDF (`pypdf`) — and handle multi-page documents.
2. **Chunk** text into LLM-sized pieces, respecting sentence boundaries.
3. Use **structured-output prompting** to extract typed fields.
4. **Validate** the model's output with JSON Schema and Pydantic-style checks.
5. Compute **field-level accuracy** against a labelled set.
6. Aggregate extracted records into a pandas DataFrame for downstream analysis.
7. Recognise document-processing pitfalls (low-confidence fields, missing data, table extraction).

## ✅ Prerequisites

Notebook 11 (LLM workflows), 15 (retrieval). Helpful: NB4 (JSON), NB5 (pandas), NB16 (tools).

## 1. The shape of every document-processing pipeline

```
   ┌──────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
   │  Source  │ ──► │ Extract  │ ──► │  Chunk   │ ──► │  LLM     │ ──► │ Validate │
   │ (PDF /…) │     │  text    │     │ (if big) │     │ extract  │     │ + collect│
   └──────────┘     └──────────┘     └──────────┘     └──────────┘     └──────────┘
                                                                            │
                                                                            ▼
                                                                       DataFrame
```

Each box is replaceable. **Extract** could be `pypdf`, `pdfplumber`, `docx2txt`, or an OCR service. **Chunk** depends on document size. **LLM** is whichever model you trust. **Validate** is your insurance against hallucination.

The pipeline is *general* — the same shape works for invoices, contracts, lab reports, anything.

## 2. Setup

In [ ]:
import json
import re
import pandas as pd
from pathlib import Path
from datetime import date

# Reuse the MockLLM pattern (vendored for this notebook's self-containment)
class MockLLM:
    """An offline mock that extracts structured invoice fields with regex."""
    def chat(self, messages, **_):
        user_text = next((m["content"] for m in reversed(messages)
                          if m["role"] == "user"), "")
        return {"text": json.dumps(self._extract(user_text))}

    @staticmethod
    def _extract(text: str) -> dict:
        # Crude but deterministic: regex-pluck the fields a real LLM would identify
        vendor = re.search(r"From:\s+(.+)", text)
        invoice_id = re.search(r"Invoice\s*#\s*([\w-]+)", text, re.IGNORECASE)
        due_date = re.search(r"Due\s+date:\s*(\d{4}-\d{2}-\d{2})", text, re.IGNORECASE)
        # \bTOTAL\b without IGNORECASE → uppercase only, so "Subtotal" doesn't match
        total = re.search(r"\bTOTAL\b[:\s]+\$?([\d,]+\.\d{2})", text)
        # Line items: lines starting with "- " or numbered "N. "
        items = []
        for line in text.splitlines():
            m = re.match(r"\s*[-\d\.]+\s+(.+?)\s+\$([\d,]+\.\d{2})\s*$", line)
            if m:
                items.append({"description": m.group(1).strip(),
                               "amount": float(m.group(2).replace(",", ""))})
        return {
            "vendor":     vendor.group(1).strip()     if vendor     else None,
            "invoice_id": invoice_id.group(1).strip() if invoice_id else None,
            "due_date":   due_date.group(1).strip()   if due_date   else None,
            "total":      float(total.group(1).replace(",", "")) if total else None,
            "line_items": items,
        }


llm = MockLLM()
print("MockLLM (invoice-extractor) ready ✅")


> 🔌 **Using a real provider (OpenAI / Anthropic / Gemini / Ollama).**
> Every notebook in this module uses an offline `MockLLM` by default so you can run them without internet or API keys. Once you want real intelligence in the answers, swap one line — the unified interface in [`llm_providers.py`](../llm_providers.py) lets you use **OpenAI**, **Anthropic** (Claude), **Google** (Gemini), or a **local model via Ollama** without changing anything else.
> See the [LLM Providers Guide](./A1_llm_providers_guide.ipynb) for the swap-in instructions, model recommendations, and cost estimates.


## 3. Generate a small set of synthetic invoices

In a real project this folder would contain PDFs from your accounting system. We synthesise plain-text invoices so the notebook runs everywhere.

In [ ]:
INVOICES = [
    """Invoice #INV-1042

From: Acme Cloud Solutions
To: Customer Inc.
Due date: 2024-05-15

Line items
- AWS hosting (April)           $1,240.00
- CDN bandwidth                 $345.50
- DDoS protection add-on        $99.00

Subtotal                        $1,684.50
Tax (7%)                        $117.92
TOTAL                           $1,802.42

Payment terms: Net 30.""",

    """Invoice #INV-1043

From: Beta Office Supplies
To: Customer Inc.
Due date: 2024-05-20

1. 24 boxes A4 paper            $144.00
2. 6 toner cartridges           $522.00
3. Office coffee subscription   $89.00

TOTAL                           $755.00

Payment terms: Net 14.""",

    """Invoice #INV-1044

From: Cygnus Marketing
To: Customer Inc.
Due date: 2024-06-01

Line items
- Google Ads management         $2,400.00
- Landing page redesign         $1,800.00
- A/B-test setup                $450.00

TOTAL                           $4,650.00""",

    """Invoice #INV-1045

From: Delta Catering
To: Customer Inc.
Due date: 2024-05-25

1. Team lunch (40 people)       $580.00
2. Coffee & snacks               $120.00

TOTAL                           $700.00""",

    """Invoice #INV-1046

From: Acme Cloud Solutions
To: Customer Inc.
Due date: 2024-06-15

Line items
- AWS hosting (May)             $1,290.00
- CDN bandwidth                 $410.00

Subtotal                        $1,700.00
Tax (7%)                        $119.00
TOTAL                           $1,819.00""",
]
print(f"Synthesised {len(INVOICES)} invoices.")
print("\n--- Sample (first invoice) ---")
print(INVOICES[0])


> 💡 **In the real world.** To get the text out of an actual PDF you'd use:
> ```python
> from pypdf import PdfReader
> text = "\n".join(p.extract_text() for p in PdfReader("inv_1042.pdf").pages)
> ```
> `pypdf` is plain-Python and works on most digital PDFs. For scanned PDFs you need OCR (`pytesseract`, `Azure Document Intelligence`, `AWS Textract`, etc.). The rest of the pipeline is identical.

## 4. Chunking — when documents are too big for one prompt

Modern LLMs handle 100K+ tokens, but you still want to chunk for three reasons:

1. **Cost** — you pay per token. Smaller prompts = lower bills.
2. **Quality** — models pay more attention to short contexts.
3. **Granular results** — chunk-level errors are localised.

Three common strategies:

In [ ]:
def chunk_by_chars(text: str, max_chars: int = 800, overlap: int = 100) -> list[str]:
    """Simple fixed-size character chunks with overlap."""
    if max_chars <= overlap:
        raise ValueError("max_chars must be larger than overlap")
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        if end >= len(text):              # reached the end → done
            break
        start = end - overlap             # otherwise step forward, leaving overlap
    return chunks


def chunk_by_paragraphs(text: str, max_chars: int = 800) -> list[str]:
    """Respect paragraph boundaries; merge small paragraphs up to max_chars."""
    paragraphs = re.split(r"\n\s*\n", text)
    chunks, buf = [], ""
    for p in paragraphs:
        if len(buf) + len(p) + 2 <= max_chars:
            buf = (buf + "\n\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            buf = p
    if buf:
        chunks.append(buf)
    return chunks


# A small long document for the demo
demo = INVOICES[0] + "\n\n" + INVOICES[1]
print(f"Document is {len(demo)} characters.")
print(f"chunk_by_chars(max=400, overlap=50): {len(chunk_by_chars(demo, 400, 50))} chunks")
print(f"chunk_by_paragraphs(max=400):         {len(chunk_by_paragraphs(demo, 400))} chunks")


> 🎯 **Pick your splitter for the data.** Fixed-char chunks are simple but split words and paragraphs. Paragraph-aware chunks keep semantic units intact at the cost of a more complex implementation. For *highly structured* documents (invoices, forms), it's usually best to keep the whole document together — they're short enough.

## 5. Extracting structured fields with the LLM

In [ ]:
EXTRACTION_SYSTEM = '''You extract structured invoice fields.

Return a JSON object with exactly these keys:
- "vendor":     string  (sender name)
- "invoice_id": string  (e.g. "INV-1042")
- "due_date":   string  (YYYY-MM-DD)
- "total":      number  (the TOTAL amount, dollars)
- "line_items": array of {"description": string, "amount": number}

If a field is missing in the document, set it to null.
Return ONLY the JSON. No prose, no markdown fences.'''


def extract_invoice(text: str) -> dict | None:
    """Call the LLM with one invoice and return the parsed JSON (or None)."""
    resp = llm.chat(messages=[
        {"role": "system", "content": EXTRACTION_SYSTEM},
        {"role": "user",   "content": text},
    ])
    try:
        return json.loads(resp["text"])
    except json.JSONDecodeError:
        return None


# Try it on the first invoice
fields = extract_invoice(INVOICES[0])
print(json.dumps(fields, indent=2))


## 6. Validating the extracted fields

The model is fast but not perfect. **Always validate** before pushing the data to your database.

In [ ]:
# Required field types — a tiny JSON-Schema-like spec
SCHEMA = {
    "vendor":     {"type": str, "required": True},
    "invoice_id": {"type": str, "required": True},
    "due_date":   {"type": str, "required": True, "pattern": r"\d{4}-\d{2}-\d{2}"},
    "total":      {"type": (int, float), "required": True, "min": 0, "max": 1_000_000},
    "line_items": {"type": list, "required": True},
}

def validate(record: dict) -> list[str]:
    """Return a list of validation errors; empty list means OK."""
    if record is None:
        return ["record is None"]
    errs = []
    for field, rule in SCHEMA.items():
        v = record.get(field)
        if v is None:
            if rule.get("required"):
                errs.append(f"missing field: {field}")
            continue
        if not isinstance(v, rule["type"]):
            errs.append(f"{field}: expected {rule['type']}, got {type(v).__name__}")
            continue
        if "pattern" in rule and not re.fullmatch(rule["pattern"], v):
            errs.append(f"{field}: '{v}' does not match {rule['pattern']!r}")
        if "min" in rule and v < rule["min"]:
            errs.append(f"{field}: {v} below min {rule['min']}")
        if "max" in rule and v > rule["max"]:
            errs.append(f"{field}: {v} above max {rule['max']}")
    return errs


for i, inv in enumerate(INVOICES):
    rec  = extract_invoice(inv)
    errs = validate(rec)
    status = "✓" if not errs else f"✗ ({'; '.join(errs)})"
    print(f"Invoice {i+1}: {status}")


> 💡 **In production** you'd use **Pydantic** (`from pydantic import BaseModel`) for this. It generates the JSON schema automatically and gives you typed objects. The hand-rolled validator above is functionally the same idea.

## 7. Building the final DataFrame

In [ ]:
# Extract every invoice and collect them
records = []
for inv in INVOICES:
    rec = extract_invoice(inv)
    if rec is not None:
        records.append(rec)

# Flatten the top-level fields into a DataFrame
df = pd.DataFrame([{k: r.get(k) for k in ("invoice_id", "vendor", "due_date", "total")}
                   for r in records])
df["due_date"] = pd.to_datetime(df["due_date"], errors="coerce")
print(df)


In [ ]:
# Vendor-level summary — the actual business question
summary = (df.groupby("vendor")
             .agg(n_invoices=("invoice_id", "count"),
                  total_billed=("total",      "sum"),
                  next_due=("due_date",      "min"))
             .sort_values("total_billed", ascending=False)
             .round(2))
print(summary)


**Read that.** Five free-form text invoices became a tidy ranked report in roughly 30 lines of code. The next step (not shown) is a plot, an email, or a write to a database — and you already know how to do all three from earlier notebooks.

## 8. Line-item drill-down

The `line_items` field is itself a list of dicts. Flattening a list-of-lists into a long DataFrame is a frequent pattern.

In [ ]:
# Build a "long" line-items table — one row per item, with the invoice_id
items = []
for r in records:
    for li in r.get("line_items", []):
        items.append({"invoice_id": r["invoice_id"],
                      "vendor":     r["vendor"],
                      "description": li["description"],
                      "amount":      li["amount"]})

items_df = pd.DataFrame(items)
print(f"Total line items: {len(items_df)}")
print(items_df.head(10))


In [ ]:
# Top categories by spend (naive: just sort by amount)
print("\nTop-5 most expensive line items:")
print(items_df.sort_values("amount", ascending=False).head().reset_index(drop=True))


## 9. Field-level accuracy — measuring extraction quality

You can't trust extraction on faith. With even a few hand-labelled documents you can measure **field-level accuracy** — the share of (document, field) cells where the model agreed with the ground truth.

In [ ]:
# Ground truth for our small set
GROUND_TRUTH = [
    {"invoice_id": "INV-1042", "vendor": "Acme Cloud Solutions",   "due_date": "2024-05-15", "total": 1802.42},
    {"invoice_id": "INV-1043", "vendor": "Beta Office Supplies",   "due_date": "2024-05-20", "total":  755.00},
    {"invoice_id": "INV-1044", "vendor": "Cygnus Marketing",       "due_date": "2024-06-01", "total": 4650.00},
    {"invoice_id": "INV-1045", "vendor": "Delta Catering",         "due_date": "2024-05-25", "total":  700.00},
    {"invoice_id": "INV-1046", "vendor": "Acme Cloud Solutions",   "due_date": "2024-06-15", "total": 1819.00},
]

# Compute per-field accuracy
fields_to_check = ["invoice_id", "vendor", "due_date", "total"]
correct = {f: 0 for f in fields_to_check}
for pred, gold in zip(records, GROUND_TRUTH):
    for f in fields_to_check:
        if pred.get(f) == gold[f]:
            correct[f] += 1

n = len(GROUND_TRUTH)
print("Field-level accuracy:")
for f, n_ok in correct.items():
    print(f"  {f:<12}: {n_ok}/{n}  ({n_ok/n:.0%})")


**On a real corpus** you'd label 50–200 documents and measure each field separately. The metric matters because *the worst field sets the floor for the whole pipeline*. A vendor name that's wrong 20% of the time will quietly corrupt downstream analytics until someone notices.

## 10. Common pitfalls

| Pitfall | Symptom | Fix |
|---|---|---|
| The LLM "hallucinates" a missing field | wrong but plausible value | Use `"if missing, set to null"` in the system prompt + validate |
| Numbers come back as strings | `"1,802.42"` instead of `1802.42` | Cast in the validator; normalise commas |
| Dates in mixed formats | `"15/5/2024"`, `"May 15, 2024"`, ... | Force `YYYY-MM-DD` in the prompt; use `pd.to_datetime(..., errors='coerce')` |
| Tables in scanned PDFs | OCR garbles columns | Use a vision-capable model or a table-aware extractor (`pdfplumber`, `camelot`) |
| Sensitive data (PII) | leaks to a third-party LLM | Run a local model, or redact before sending |
| Inconsistent vendor names | "Acme Inc.", "Acme, Inc", "ACME INC" | Add a vendor-normalisation step after extraction |

## 🧪 Practice exercises

### Exercise 1 — Add a `subtotal` field

Modify the extraction prompt and validator to also pull a `subtotal` (the pre-tax amount). For invoices that don't show a separate subtotal, the field should be `null`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# 1. Update the system prompt to mention subtotal
EXTRACTION_SYSTEM_V2 = EXTRACTION_SYSTEM.replace(
    '- "line_items"',
    '- "subtotal":   number  (pre-tax subtotal, null if not shown)\n- "line_items"',
)

# 2. Update the schema
SCHEMA["subtotal"] = {"type": (int, float), "required": False, "min": 0}

# 3. Update the MockLLM extractor too — in production this prompt change is all you need
def extract_with_subtotal(text):
    sub = re.search(r"Subtotal[:\s]+\$?([\d,]+\.\d{2})", text, re.IGNORECASE)
    rec = MockLLM._extract.__func__(text)
    rec["subtotal"] = float(sub.group(1).replace(",", "")) if sub else None
    return rec

for inv in INVOICES:
    print(extract_with_subtotal(inv)["subtotal"])
```

In a real implementation you'd just update the prompt and let the LLM figure it out. The mock needs explicit code because it's not an LLM.
</details>

### Exercise 2 — Vendor normalisation

The vendor field can have inconsistent spellings (`"Acme Inc."` vs `"Acme, Inc"` vs `"ACME INC"`). Write `normalize_vendor(name)` that returns a canonical lowercase form with commas and periods removed, and use it before grouping.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def normalize_vendor(name):
    if not isinstance(name, str):
        return name
    n = name.strip().lower()
    n = re.sub(r"[.,]", "", n)        # drop punctuation
    n = re.sub(r"\s+", " ", n)        # collapse whitespace
    return n.title()                  # capitalise for display

df["vendor_canonical"] = df["vendor"].map(normalize_vendor)
print(df[["vendor", "vendor_canonical"]])
```

This is a small but high-leverage step: a 5% inconsistency rate becomes a 95% grouping error if you skip it.
</details>

### Exercise 3 — Confidence-aware extraction

Modify `extract_invoice` to also return a `confidence` score: simply the fraction of required fields that came back non-null. Then list any invoice with confidence below 1.0 for human review.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def extract_with_confidence(text):
    rec = extract_invoice(text) or {}
    required = ["vendor", "invoice_id", "due_date", "total"]
    n_ok = sum(1 for f in required if rec.get(f) is not None)
    rec["confidence"] = n_ok / len(required)
    return rec

flagged = []
for inv in INVOICES:
    rec = extract_with_confidence(inv)
    if rec["confidence"] < 1.0:
        flagged.append(rec)

print(f"{len(flagged)} invoices need human review.")
```

**Why this matters.** Pushing only the high-confidence records through automatically — and routing the rest to a human — is how production document pipelines stay accurate at scale. The "automation rate" is the share of documents you can process without human touch; raising it 5% is often worth a quarter's engineering effort.
</details>

### Exercise 4 — Debug me 🐞

The function below tries to compute the sum of line items per invoice and check it against `total`, but it sometimes returns `False` for valid invoices. Find the bug.

```python
def line_items_match_total(rec):
    items_sum = sum(li["amount"] for li in rec["line_items"])
    return items_sum == rec["total"]

for r in records:
    print(r["invoice_id"], line_items_match_total(r))
```

In [ ]:
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

The comparison uses `==` on floats — which fails for sums-of-amounts whose total includes tax (so `items_sum < total`). Two issues at once:

1. **Float equality is unreliable** — use a tolerance.
2. **The total is *with tax*** for some invoices, so `items_sum != total` is expected unless we subtract tax.

```python
def line_items_match_total(rec, tolerance=0.01):
    items_sum = sum(li["amount"] for li in rec["line_items"])
    if rec["total"] is None:
        return None
    diff = abs(items_sum - rec["total"])
    return diff < tolerance or diff < 0.1 * rec["total"]    # allow up to 10% (tax)

for r in records:
    print(r["invoice_id"], line_items_match_total(r))
```

**Lesson.** Always think about *float tolerance* and *what the number actually represents* before comparing. The bug isn't only `==` on floats; it's *misunderstanding the business invariant*.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — Per-field confidence flags

Modify `extract_invoice` so the returned record includes a `confidence` field: `'high'` if all required fields are present and pass validation, `'low'` otherwise. Then build a DataFrame showing how many invoices are flagged low.


<details>
<summary>💡 <b>Solution</b></summary>

```python
def extract_invoice_with_conf(text):
    rec = extract_invoice(text) or {}
    errs = validate(rec)
    rec["confidence"] = "high" if not errs else "low"
    rec["errors"]     = errs
    return rec


rows = [extract_invoice_with_conf(t) for t in INVOICES]
df = pd.DataFrame([{"invoice_id": r.get("invoice_id"),
                     "vendor":    r.get("vendor"),
                     "confidence": r["confidence"],
                     "errors":    r["errors"]} for r in rows])
print(df)
print(f"\nLow-confidence: {(df['confidence']=='low').sum()}/{len(df)}")
```

**Why confidence per record matters.** It lets the downstream
system *route* — auto-process the high-confidence ones, send the
low-confidence ones to a human. This is how document AI economics
work in practice: optimise the *automation rate*, not the
model's raw accuracy.

</details>

### Stretch exercise B — Multi-page invoice — chunked extraction

Suppose an invoice text is 4× the size of our samples. Use `chunk_by_paragraphs` to split it, run `extract_invoice` on each chunk, and merge the results — keeping the *first non-null* value per field.


<details>
<summary>💡 <b>Solution</b></summary>

```python
big_invoice = "\n\n".join(INVOICES[:3])      # pretend this is one long doc

chunks = chunk_by_paragraphs(big_invoice, max_chars=400)
print(f"Split into {len(chunks)} chunks.")

merged = {"vendor": None, "invoice_id": None, "due_date": None, "total": None, "line_items": []}
for c in chunks:
    part = extract_invoice(c) or {}
    for k in ("vendor", "invoice_id", "due_date", "total"):
        if merged[k] is None and part.get(k) is not None:
            merged[k] = part[k]
    merged["line_items"].extend(part.get("line_items", []))

print(json.dumps(merged, indent=2, default=str)[:400])
```

**The "first non-null wins" merge** is the standard pattern for
chunked extraction. For numeric fields you might *sum* across
chunks instead (total amount = sum of per-chunk totals). For
line items you almost always *concatenate*.

</details>

## 🎁 Bonus mini-project — A "monthly billing report"

Combine everything into a single function `monthly_billing_report(invoices)` that:

1. Extracts each invoice into a record.
2. Validates every record (prints which invoices need review).
3. Returns two DataFrames: `summary_by_vendor` and `line_items_long`.
4. Prints a "month total" — the sum of all `total`s, formatted as currency.

You should be able to call it on the global `INVOICES` and get a polished report.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def monthly_billing_report(invoices):
    records, problems = [], []
    for i, text in enumerate(invoices):
        rec = extract_invoice(text)
        errs = validate(rec)
        if errs:
            problems.append((i, errs))
            continue
        records.append(rec)

    # By-vendor summary
    df = pd.DataFrame([{k: r.get(k) for k in ("invoice_id", "vendor", "due_date", "total")}
                       for r in records])
    df["due_date"] = pd.to_datetime(df["due_date"], errors="coerce")
    by_vendor = (df.groupby("vendor")
                   .agg(n_invoices=("invoice_id","count"),
                        total_billed=("total","sum"))
                   .sort_values("total_billed", ascending=False)
                   .round(2))

    # Long line items
    items = []
    for r in records:
        for li in r.get("line_items", []):
            items.append({"invoice_id": r["invoice_id"], "vendor": r["vendor"],
                          "description": li["description"], "amount": li["amount"]})
    items_long = pd.DataFrame(items)

    print(f"📋 Processed {len(records)} invoices  ({len(problems)} need review)")
    print(f"💰 Month total: ${df['total'].sum():,.2f}\n")
    return by_vendor, items_long


bv, il = monthly_billing_report(INVOICES)
print("--- Summary by vendor ---")
print(bv)
print("\n--- Sample of line items ---")
print(il.head())
```

**What you just built.** A complete document-processing pipeline — the same one a finance ops team would deploy to automate AP. The MockLLM is doing the regex work; in production a real LLM does it *correctly* across thousands of invoice templates.
</details>

## 🧠 Key takeaways

1. **Document processing = extract → chunk → LLM → validate → aggregate.** The shape is universal.
2. **`pypdf` for digital PDFs**, OCR (`pytesseract`, hosted services) for scans.
3. Chunk by paragraph or sentence when documents are big; keep small docs whole.
4. **Always specify JSON output** explicitly in the system prompt — and always wrap `json.loads` in `try/except`.
5. **Validate every record** with a schema. Use Pydantic in production; a hand-rolled validator works for small projects.
6. **Field-level accuracy** is your honest metric. The worst field sets the pipeline's floor.
7. **Confidence-routing** (auto for high-confidence, human for low) is what makes document AI economical.
8. Vendor / category normalisation matters more than the model — a 5% spelling drift kills your group-bys.

## ✅ Self-assessment

- [ ] Extract text from a PDF (or text file) into a Python string
- [ ] Chunk a long document by paragraph
- [ ] Write a structured-output system prompt that returns valid JSON
- [ ] Validate the extracted fields with a JSON-Schema-like check
- [ ] Compute field-level accuracy against ground truth
- [ ] Aggregate extracted records into a tidy DataFrame
- [ ] Route low-confidence records for human review

## 🚀 Next step

Continue with **Notebook 18 — From Notebook to Project**, where you take the toolkit you've been writing — `MockLLM`, `extract_invoice`, the validators — and package it into a proper Python project with tests, an entry point, and a virtualenv.